# Практическое занятие 12. Предварительная обработка текстовых данных: токенизация, нормализация и удаление стоп-слов

**Технологии больших данных (Big Data)**, 6 семестр.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mkb40-svg/big_data_6semestr/blob/main/notebooks/p12.ipynb)

**Цели занятия:**

- Освоить шаги предобработки текста: нормализация апострофов и регистра, удаление пунктуации, токенизация, деление на предложения
- Научиться удалять стоп-слова узбекского языка и оценивать, как предобработка меняет объём данных
- Собрать переиспользуемую функцию-конвейер `preprocess(text)` и простой стеммер для агглютинативной морфологии

> Автоматическая проверка заданий доступна в веб-тренажёре. В блокноте выполняйте код и сверяйте результат с условием.

In [ ]:
# ▶ Подготовка среды: выполните эту ячейку первой.
# Загружает учебные данные из репозитория тренажёра и запускает локальный веб-сервер
# (нужен для заданий с requests: учебный сайт SITE и учебный API).
import os, sys, subprocess, time

REPO = "https://github.com/mkb40-svg/big_data_6semestr.git"
if not os.path.exists("data/manifest.json"):
    if "OWNER/REPO" in REPO:
        raise SystemExit("Укажите в REPO адрес репозитория тренажёра на GitHub, например https://github.com/<логин>/bigdata-trainer.git")
    if not os.path.exists("bigdata-trainer"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "bigdata-trainer"], check=True)
    os.chdir("bigdata-trainer")

subprocess.Popen([sys.executable, "-m", "http.server", "8765"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)
BASE_URL = "http://localhost:8765/"
SITE = BASE_URL + "data/site/"
API = BASE_URL + "data/api/v1/"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openpyxl", "wordcloud", "seaborn"])
print("Готово. Рабочая папка:", os.getcwd())

## Теория

## Зачем нужна предобработка

Компьютер видит текст как последовательность символов. Для него «Oʻzbekiston», «O'zbekiston» и «OʻZBEKISTON» — три разных строки, а «kitob.» и «kitob» — разные слова. **Предобработка** приводит текст к единому виду, чтобы подсчёты частот, поиск и машинное обучение работали с *языковыми* единицами, а не со случайностями набора.

Типичный конвейер (pipeline):

1. **очистка** — HTML-теги, ссылки, эмодзи (подробно — в Л5);
2. **нормализация** — единые апострофы, Unicode-форма NFC, регистр;
3. **сегментация** — деление на предложения;
4. **токенизация** — деление на слова;
5. **фильтрация** — стоп-слова, числа, слишком короткие токены;
6. **стемминг/лемматизация** — сведение словоформ к основе.

## Нормализация апострофов

| Знак | Код | Где встречается |
|---|---|---|
| ʻ | U+02BB | канон: **oʻ**, **gʻ** |
| ʼ | U+02BC | канон: *tutuq belgisi* — taʼlim, maʼlumot |
| ' | U+0027 | клавиатура (ASCII) |
| ‘ ’ | U+2018, U+2019 | автозамена в текстовых редакторах |
| обратный апостроф | U+0060 | «быстрый» набор на клавиатуре |

Правило: после **o** и **g** любой вариант превращается в ʻ, все остальные — в ʼ.

```python
import re
def normalize(text):
    text = re.sub(r"([oOgG])['\u2018\u2019\u0060]", lambda m: m.group(1) + "\u02bb", text)
    return re.sub(r"['\u2018\u2019\u0060]", "\u02bc", text)

normalize("O'zbekiston ta’limi")     # 'Oʻzbekiston taʼlimi'
```

> **Внимание: Ограничение правила**  
> Знак после **o/g** может быть и *tutuq belgisi*: «moʼjiza», «moʼtadil». Простое правило запишет их как «moʻjiza». Точная нормализация требует словаря исключений — это типичная ситуация в обработке языка: правила дают 95–99 %, остальное — словари и статистика.

Для «невидимых» различий Unicode (одна буква может быть записана одним или двумя кодами) используйте `unicodedata.normalize("NFC", text)`.

**Регистр**: `text.lower()`. Для узбекской латиницы проблем нет, кириллица тоже понижается корректно.

## Токенизация регулярными выражениями

| Шаблон | Значение |
|---|---|
| `\w` | буква, цифра или `_` (с учётом Unicode, включая ʻ и ʼ) |
| `\s` | пробельный символ (пробел, табуляция, перевод строки) |
| `[^\w\s]` | всё, что не буква и не пробел, — пунктуация |
| `\w+(?:-\w+)*` | слово, возможно с дефисами: *IT-park*, *qoʻllab-quvvatlash* |
| `(?<=[.!?])\s+` | пробелы **после** знака конца предложения (просмотр назад) |

```python
text = "Bino zamonaviy texnologiyalar bilan jihozlangan bo'lib, unda IT-park bor."
text.split()                          # [..., "bo'lib,", ..., 'bor.'] — пунктуация прилипла
re.findall(r"\w+", text.lower())      # [..., 'bo', 'lib', ..., 'it', 'park', 'bor'] — апостроф порвал слово
re.findall(r"\w+", normalize(text).lower())   # [..., 'boʻlib', ...] — верно
```

**Предложения** делим по `. ! ?`, за которыми идёт пробел, и по переводам строк:

```python
sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+|\n+", text) if s.strip()]
```

Сокращения («va h.k.», «t.y.») и инициалы ломают такое правило — в серьёзных системах используют обученные сегментаторы.

## Стоп-слова

**Стоп-слова** — частые служебные слова (*va, bilan, uchun, bu, ham*), которые почти не несут темы текста. Их удаляют перед частотным анализом, поиском ключевых слов, тематическим моделированием.

```python
with open("data/stopwords_uz.txt", encoding="utf-8") as f:
    stopwords = {w.strip() for w in f if w.strip()}      # множество: проверка "in" за O(1)
content = [t for t in tokens if t not in stopwords]
```

> **Заметка: Список стоп-слов нормализован**  
> В файле слова записаны канонически (*boʻlib, koʻp*). Если текст не нормализован, «bo'lib» не найдётся в списке и останется в данных.

> **Совет: Не удаляйте стоп-слова вслепую**  
> Для анализа тональности слово **emas** («не») критично: «yaxshi emas» — отрицательный отзыв. Для n-грамм и синтаксического анализа стоп-слова тоже нужны.

## Стемминг агглютинативных слов

Узбекское слово — это корень + цепочка суффиксов: *kitob + lar + imiz + dan* = «из наших книг». **Стеммер** отрезает известные суффиксы и оставляет основу:

```python
SUFFIXES = ["lar", "ning", "dan", "dagi", "imiz"]
def stem(word):
    changed = True
    while changed:                                   # повторяем, пока что-то отрезается
        changed = False
        for suf in sorted(SUFFIXES, key=len, reverse=True):   # сначала длинные
            if word.endswith(suf) and len(word) - len(suf) >= 3:
                word, changed = word[:-len(suf)], True
                break
    return word

stem("kitoblarimizdan")     # 'kitob'
```

Ошибки стеммера: **перестемминг** (*maydoni → maydo*: «ni» отрезано у корня *maydon*) и **недостемминг** (редкие суффиксы не учтены). Точнее работают морфологические анализаторы и **лемматизаторы**, опирающиеся на словарь корней.

## Задание 1. Нормализация апострофов

Напишите функцию `normalize_apostrophes(text)`:
1. после букв **o, O, g, G** любой из вариантов `' ‘ ’` или обратный апостроф заменяется на ʻ (U+02BB);
2. все остальные такие знаки заменяются на ʼ (U+02BC);
3. канонические ʻ и ʼ остаются без изменений.

Затем посчитайте, какие апострофы встречаются в корпусе (таблица `df` уже загружена): `before` — `Counter` символов из `ALL_APOS` по всем текстам **до** нормализации, `after` — то же **после**.

<details><summary>💡 Подсказка</summary>

Вторым аргументом `re.sub` может быть функция: `lambda m: m.group(1) + OK`. Второй шаг: `re.sub("[" + VARIANTS + "]", TUTUQ, text)`. Для `after` нормализуйте каждый текст перед подсчётом.

</details>

In [ ]:
# Подготовительный код задания
import re
import pandas as pd
from collections import Counter

df = pd.read_csv("data/corpus_uz/meta.csv")[["file", "category"]]
df["text"] = [open("data/corpus_uz/" + f, encoding="utf-8").read() for f in df["file"]]

In [ ]:
OK = "\u02bb"       # ʻ — для oʻ, gʻ
TUTUQ = "\u02bc"    # ʼ — tutuq belgisi
VARIANTS = "'\u2018\u2019\u0060"     # ' ‘ ’ и обратный апостроф
ALL_APOS = VARIANTS + OK + TUTUQ

def normalize_apostrophes(text):
    text = re.sub("([oOgG])[" + VARIANTS + "]", ..., text)
    text = ...
    return text

print(normalize_apostrophes("O'zbekiston, g\u0060alaba, so\u2018nggi, ta\u2019lim, ma'lumot"))

before = Counter(ch for text in df["text"] for ch in text if ch in ALL_APOS)
after = ...
print(before)
print(after)

## Задание 2. Три способа токенизации

В переменной `text` — новость `uz_01.txt` (апострофы ASCII). Функция `normalize` уже определена. Сравните способы токенизации:

- `tokens_split` — `text.split()`;
- `tokens_raw` — `re.findall(r"\w+", text.lower())` (без нормализации);
- `tokens_norm` — `\w+` по `normalize(text).lower()`;
- `tokens_hyph` — шаблон `\w+(?:-\w+)*` по `normalize(text).lower()` (слова с дефисом не рвутся);
- `broken` — **множество** «обрывков»: токены из `tokens_raw`, которых нет среди `tokens_norm`.

Выведите длины всех списков и `broken`.

<details><summary>💡 Подсказка</summary>

Множество обрывков: `set(tokens_raw) - set(tokens_norm)`. Не забудьте `.lower()` во всех вариантах, кроме `split`.

</details>

In [ ]:
# Подготовительный код задания
import re
import pandas as pd
from collections import Counter

def normalize(text):
    """Все варианты апострофа -> канонические: oʻ/gʻ (U+02BB) и ʼ (U+02BC)."""
    text = re.sub(r"([oOgG])['\u2018\u2019\u0060]", lambda m: m.group(1) + "\u02bb", text)
    return re.sub(r"['\u2018\u2019\u0060]", "\u02bc", text)

text = open("data/corpus_uz/uz_01.txt", encoding="utf-8").read()

In [ ]:
tokens_split = ...
tokens_raw = ...
tokens_norm = ...
tokens_hyph = ...
broken = ...

for name, toks in [("split", tokens_split), ("raw", tokens_raw), ("norm", tokens_norm), ("hyph", tokens_hyph)]:
    print(name, len(toks), toks[:8])
print("Обрывки:", sorted(broken))

## Задание 3. Пунктуация и предложения

Напишите две функции:

1. `remove_punct(text)` — заменяет каждый символ, который не является буквой/цифрой и не пробел (шаблон `[^\w\s]`), на пробел, затем схлопывает все пробельные символы в один пробел и убирает пробелы по краям.
   Пример: `remove_punct('"Salom", dedi u. Oʻzbek-tili!')` → `'Salom dedi u Oʻzbek tili'`.
2. `split_sentences(text)` — список предложений: деление после `. ! ?` с пробелами и по переводам строк; знак конца остаётся в предложении, пробелы по краям убраны, пустых элементов нет.
   Пример: `"Salom! Qalaysiz? Yaxshi.\n\nRahmat"` → `['Salom!', 'Qalaysiz?', 'Yaxshi.', 'Rahmat']`.

Примените `split_sentences` к тексту `uz_05` (переменная `text`, уже нормализован): сохраните список в `sents`, а самое длинное по числу токенов предложение — в `longest`.

<details><summary>💡 Подсказка</summary>

Пробелы: `re.sub(r"\s+", " ", text)`. Предложения: `re.split(r"(?<=[.!?])\s+|\n+", text)`, затем `[s.strip() for s in parts if s.strip()]`. Самое длинное: `max(sents, key=lambda s: len(re.findall(r"\w+", s)))`.

</details>

In [ ]:
# Подготовительный код задания
import re
import pandas as pd
from collections import Counter

def normalize(text):
    """Все варианты апострофа -> канонические: oʻ/gʻ (U+02BB) и ʼ (U+02BC)."""
    text = re.sub(r"([oOgG])['\u2018\u2019\u0060]", lambda m: m.group(1) + "\u02bb", text)
    return re.sub(r"['\u2018\u2019\u0060]", "\u02bc", text)

text = normalize(open("data/corpus_uz/uz_05.txt", encoding="utf-8").read())

In [ ]:
def remove_punct(text):
    text = re.sub(..., " ", text)
    text = ...          # схлопнуть пробелы
    return text.strip()

def split_sentences(text):
    parts = re.split(..., text)
    return ...

print(remove_punct('"Salom", dedi u. O\u02bbzbek-tili!'))
print(split_sentences("Salom! Qalaysiz? Yaxshi.\n\nRahmat"))

sents = ...
longest = ...
print(len(sents), "предложений")
print("Самое длинное:", longest)

## Задание 4. Удаление стоп-слов

В `tokens` — все токены корпуса (нормализованы, в нижнем регистре).

1. Прочитайте `data/stopwords_uz.txt` (по слову в строке) в **множество** `stopwords` (без пустых строк и пробелов по краям).
2. `content` — токены без стоп-слов (порядок сохраняется).
3. `stop_share` — доля токенов корпуса, которые оказались стоп-словами.
4. `top_before` и `top_after` — 10 самых частых слов до и после удаления стоп-слов (`Counter(...).most_common(10)`).

<details><summary>💡 Подсказка</summary>

`{w.strip() for w in f if w.strip()}` — множество из строк файла. Фильтр: `[t for t in tokens if t not in stopwords]`.

</details>

In [ ]:
# Подготовительный код задания
import re
import pandas as pd
from collections import Counter

def normalize(text):
    """Все варианты апострофа -> канонические: oʻ/gʻ (U+02BB) и ʼ (U+02BC)."""
    text = re.sub(r"([oOgG])['\u2018\u2019\u0060]", lambda m: m.group(1) + "\u02bb", text)
    return re.sub(r"['\u2018\u2019\u0060]", "\u02bc", text)

df = pd.read_csv("data/corpus_uz/meta.csv")[["file", "category"]]
df["text"] = [open("data/corpus_uz/" + f, encoding="utf-8").read() for f in df["file"]]

tokens = [t for text in df["text"] for t in re.findall(r"\w+", normalize(text).lower())]

In [ ]:
with open("data/stopwords_uz.txt", encoding="utf-8") as f:
    stopwords = ...

content = ...
stop_share = ...
top_before = ...
top_after = ...

print("Стоп-слов в списке:", len(stopwords))
print(f"Доля стоп-слов в корпусе: {stop_share:.1%}")
print("До:   ", top_before)
print("После:", top_after)

## Задание 5. Конвейер preprocess(text)

Соберите шаги в одну переиспользуемую функцию `preprocess(text)`, которая возвращает **список токенов**:

1. нормализация апострофов (`normalize` уже есть);
2. нижний регистр;
3. токенизация `\w+`;
4. удаление стоп-слов (множество `STOPWORDS` уже загружено);
5. удаление токенов короче 2 символов и токенов, состоящих только из цифр.

Примените её ко всему корпусу:
- `df["tokens"]` — результат `preprocess` для каждого текста;
- `df["n_raw"]` — число «слов» исходного текста по `text.split()`;
- `df["n_clean"]` — длина списка `tokens`;
- `reduction` — на какую долю уменьшился объём корпуса: `1 - сумма n_clean / сумма n_raw`.

<details><summary>💡 Подсказка</summary>

Внутри функции: `tokens = re.findall(r"\w+", text.lower())`, затем `[t for t in tokens if t not in STOPWORDS and len(t) >= 2 and not t.isdigit()]`. Длина списков в столбце: `df["tokens"].str.len()`.

</details>

In [ ]:
# Подготовительный код задания
import re
import pandas as pd
from collections import Counter

def normalize(text):
    """Все варианты апострофа -> канонические: oʻ/gʻ (U+02BB) и ʼ (U+02BC)."""
    text = re.sub(r"([oOgG])['\u2018\u2019\u0060]", lambda m: m.group(1) + "\u02bb", text)
    return re.sub(r"['\u2018\u2019\u0060]", "\u02bc", text)

df = pd.read_csv("data/corpus_uz/meta.csv")[["file", "category"]]
df["text"] = [open("data/corpus_uz/" + f, encoding="utf-8").read() for f in df["file"]]

with open("data/stopwords_uz.txt", encoding="utf-8") as _f:
    STOPWORDS = {w.strip() for w in _f if w.strip()}

In [ ]:
def preprocess(text):
    text = normalize(text)
    ...
    return tokens

print(preprocess("O'zbekiston va O\u02bbzbekistonda 2026-yil ham!"))

df["tokens"] = ...
df["n_raw"] = ...
df["n_clean"] = ...
reduction = ...

print(f"Объём уменьшился на {reduction:.1%}")
df[["file", "category", "n_raw", "n_clean"]]

## Задание 6. Простой стеммер для узбекского (продвинутое)

Напишите функцию `stem(word)` для списка суффиксов `SUFFIXES` (уже задан):

- найдите **самый длинный** суффикс из списка, на который оканчивается слово, при условии, что после отрезания останется **не меньше 3 символов**;
- отрежьте его и повторяйте, пока что-то отрезается.

Пример: `kitoblarimizdan → kitoblarimiz → kitoblar → kitob`.

Затем для `content_tokens` (все токены корпуса после `preprocess`):
- `stems` — список основ (`stem` от каждого токена);
- `n_types_before`, `n_types_after` — число уникальных словоформ и уникальных основ;
- `families` — словарь `{основа: отсортированный список разных словоформ}` только для основ, у которых **3 и больше** разных словоформ.

<details><summary>💡 Подсказка</summary>

Перебор: `for suf in sorted(SUFFIXES, key=len, reverse=True)`; условие: `word.endswith(suf) and len(word) - len(suf) >= 3`. Для families сначала соберите `{основа: set(словоформ)}`, затем оставьте основы с `len(forms) >= 3`.

</details>

In [ ]:
# Подготовительный код задания
import re
import pandas as pd
from collections import Counter

def normalize(text):
    """Все варианты апострофа -> канонические: oʻ/gʻ (U+02BB) и ʼ (U+02BC)."""
    text = re.sub(r"([oOgG])['\u2018\u2019\u0060]", lambda m: m.group(1) + "\u02bb", text)
    return re.sub(r"['\u2018\u2019\u0060]", "\u02bc", text)

df = pd.read_csv("data/corpus_uz/meta.csv")[["file", "category"]]
df["text"] = [open("data/corpus_uz/" + f, encoding="utf-8").read() for f in df["file"]]

with open("data/stopwords_uz.txt", encoding="utf-8") as _f:
    STOPWORDS = {w.strip() for w in _f if w.strip()}

SUFFIXES = ["lar", "ning", "ni", "ga", "da", "dan", "dagi", "lik", "chi", "si",
            "imiz", "ingiz", "moqda", "gan", "di", "adi", "ish", "ib"]

def preprocess(text):
    tokens = re.findall(r"\w+", normalize(text).lower())
    return [t for t in tokens if t not in STOPWORDS and len(t) >= 2 and not t.isdigit()]

content_tokens = [t for text in df["text"] for t in preprocess(text)]

In [ ]:
print(SUFFIXES)

def stem(word):
    changed = True
    while changed:
        changed = False
        for suf in ...:          # сначала самые длинные суффиксы
            if ...:
                word = ...
                changed = True
                break
    return word

for w in ["kitoblarimizdan", "talabalarning", "rivojlanmoqda", "dasturchilar", "maydoni"]:
    print(w, "->", stem(w))

stems = ...
n_types_before = ...
n_types_after = ...
families = ...

print("Словоформ:", n_types_before, "основ:", n_types_after)
for s, forms in sorted(families.items(), key=lambda x: -len(x[1]))[:10]:
    print(s, forms)

## Самостоятельная работа

**Тема для самостоятельной работы: «Изучение методов предварительной обработки текстовых данных».**

1. Добавьте в `preprocess` параметры `remove_stop=True`, `min_len=2`, `stem=False` и сравните объём словаря корпуса при разных настройках.
2. Найдите в корпусе не менее 10 ошибок вашего стеммера (перестемминг и недостемминг) и предложите правила или словарь исключений.
3. Сравните предобработку для двух задач — поиска ключевых слов и анализа тональности: какие шаги в них должны отличаться и почему?